# RDD2022 EDA and Cleaning
This notebook is used for:
- exploring the dataset
- checking annotation quality
- identifying data issues
- preparing cleaner YOLO-ready data

In [1]:
from pathlib import Path

PROJECT_DIR = Path.cwd().parent
DATA_DIR = PROJECT_DIR / "Data" / "RDD_SPLIT"
SORTED_DIR = PROJECT_DIR / "Data" / "RDD_SPLIT_BY_COUNTRY"

splits = ["train", "val", "test"]
countries = [
    "China_Drone",
    "China_MotorBike",
    "Czech",
    "India",
    "Japan",
    "Norway",
    "United_States"
]
image_suffixes = {".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"}

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_DIR exists:", DATA_DIR.exists())
print("SORTED_DIR exists:", SORTED_DIR.exists())
print("Splits:", splits)
print("Countries:", countries)

PROJECT_DIR: /Users/ahadm/Desktop/computer-vision-project-sabiq
DATA_DIR exists: True
SORTED_DIR exists: True
Splits: ['train', 'val', 'test']
Countries: ['China_Drone', 'China_MotorBike', 'Czech', 'India', 'Japan', 'Norway', 'United_States']


## Dataset Overview
Count images, labels, empty labels, and non-empty labels for each split.

In [2]:
overview = []

for split in splits:
    image_files = list((DATA_DIR / split / "images").iterdir())
    image_files = [p for p in image_files if p.suffix in image_suffixes]

    label_files = list((DATA_DIR / split / "labels").glob("*.txt"))

    empty_count = 0
    non_empty_count = 0

    for label_file in label_files:
        lines = [line.strip() for line in label_file.read_text().splitlines() if line.strip()]
        if len(lines) == 0:
            empty_count += 1
        else:
            non_empty_count += 1

    overview.append({
        "split": split,
        "images": len(image_files),
        "labels": len(label_files),
        "empty_labels": empty_count,
        "non_empty_labels": non_empty_count
    })

for row in overview:
    print(row)

{'split': 'train', 'images': 26869, 'labels': 26869, 'empty_labels': 8097, 'non_empty_labels': 18772}
{'split': 'val', 'images': 5758, 'labels': 5758, 'empty_labels': 1837, 'non_empty_labels': 3921}
{'split': 'test', 'images': 5758, 'labels': 5758, 'empty_labels': 1790, 'non_empty_labels': 3968}


## Empty Label Ratio
Measure the proportion of images with no annotated damage objects in each split.

In [ ]:
for row in overview:
    total = row["labels"]
    empty_ratio = row["empty_labels"] / total if total else 0
    non_empty_ratio = row["non_empty_labels"] / total if total else 0

    print(f"{row['split'].upper()}:")
    print(f"empty ratio     = {empty_ratio:.2%}")
    print(f"non-empty ratio = {non_empty_ratio:.2%}")

TRAIN:
  empty ratio     = 30.14%
  non-empty ratio = 69.86%
VAL:
  empty ratio     = 31.90%
  non-empty ratio = 68.10%
TEST:
  empty ratio     = 31.09%
  non-empty ratio = 68.91%


## Label Quality Summary
Check annotation validity, empty labels, bbox constraints, and discovered classes.

In [4]:
all_label_files = list(SORTED_DIR.glob("*/*/labels/*.txt"))

empty_files = []
bad_format = []
bad_class = []
bad_bbox = []
class_ids_found = set()
total_boxes = 0

for label_file in all_label_files:
    lines = [line.strip() for line in label_file.read_text().splitlines() if line.strip()]

    if len(lines) == 0:
        empty_files.append(label_file)
        continue

    for line_num, line in enumerate(lines, start=1):
        parts = line.split()

        if len(parts) != 5:
            bad_format.append((label_file, line_num, line))
            continue

        try:
            class_id = int(parts[0])
            x_center, y_center, width, height = map(float, parts[1:])
        except:
            bad_format.append((label_file, line_num, line))
            continue

        class_ids_found.add(class_id)
        total_boxes += 1

        if class_id < 0:
            bad_class.append((label_file, line_num, class_id))

        if not (0 <= x_center <= 1 and 0 <= y_center <= 1 and 0 < width <= 1 and 0 < height <= 1):
            bad_bbox.append((label_file, line_num, (x_center, y_center, width, height)))

print("Total label files:", len(all_label_files))
print("Empty label files:", len(empty_files))
print("Bad format lines:", len(bad_format))
print("Bad class IDs:", len(bad_class))
print("Bad bbox values:", len(bad_bbox))
print("Total boxes:", total_boxes)
print("Class IDs found:", sorted(class_ids_found))

Total label files: 38385
Empty label files: 11724
Bad format lines: 0
Bad class IDs: 0
Bad bbox values: 1
Total boxes: 65712
Class IDs found: [0, 1, 2, 3, 4]


## Inspect Invalid Bounding Boxes
Display the exact invalid annotation entries before deciding how to clean them.

In [5]:
if bad_bbox:
    print("Invalid bbox entries:")
    for item in bad_bbox[:10]:
        print(item)
else:
    print("No invalid bbox entries found.")

Invalid bbox entries:
(PosixPath('/Users/ahadm/Desktop/computer-vision-project-sabiq/Data/RDD_SPLIT_BY_COUNTRY/train/Japan/labels/Japan_001265.txt'), 4, (0.33, 0.790833, 0.0, 0.001667))


## Clean Invalid Bounding Box
Remove annotation lines with invalid YOLO bbox values while keeping valid lines.

In [6]:
target_file = SORTED_DIR / "train" / "Japan" / "labels" / "Japan_001265.txt"

lines = target_file.read_text().splitlines()
cleaned_lines = []
removed_lines = []

for i, line in enumerate(lines, start=1):
    line = line.strip()
    if not line:
        continue

    parts = line.split()

    if len(parts) != 5:
        removed_lines.append((i, line, "bad_format"))
        continue

    try:
        class_id = int(parts[0])
        x_center, y_center, width, height = map(float, parts[1:])
    except:
        removed_lines.append((i, line, "parse_error"))
        continue

    is_valid = (
        class_id >= 0 and
        0 <= x_center <= 1 and
        0 <= y_center <= 1 and
        0 < width <= 1 and
        0 < height <= 1
    )

    if is_valid:
        cleaned_lines.append(line)
    else:
        removed_lines.append((i, line, "invalid_bbox"))

target_file.write_text("\n".join(cleaned_lines) + ("\n" if cleaned_lines else ""))

print("File cleaned:", target_file.name)
print("Removed lines:", len(removed_lines))
print("Remaining valid lines:", len(cleaned_lines))

if removed_lines:
    print("\nRemoved entries:")
    for item in removed_lines:
        print(item)

File cleaned: Japan_001265.txt
Removed lines: 1
Remaining valid lines: 3

Removed entries:
(4, '2 0.330000 0.790833 0.000000 0.001667', 'invalid_bbox')


## Re-check Invalid Bounding Boxes
Verify that no invalid YOLO bounding boxes remain after cleaning.

In [7]:
bad_bbox_after_cleaning = []

for label_file in SORTED_DIR.glob("*/*/labels/*.txt"):
    lines = [line.strip() for line in label_file.read_text().splitlines() if line.strip()]

    for line_num, line in enumerate(lines, start=1):
        parts = line.split()
        if len(parts) != 5:
            continue

        try:
            class_id = int(parts[0])
            x_center, y_center, width, height = map(float, parts[1:])
        except:
            continue

        if not (0 <= x_center <= 1 and 0 <= y_center <= 1 and 0 < width <= 1 and 0 < height <= 1):
            bad_bbox_after_cleaning.append((label_file, line_num, (x_center, y_center, width, height)))

print("Invalid bbox entries after cleaning:", len(bad_bbox_after_cleaning))

if bad_bbox_after_cleaning[:5]:
    for item in bad_bbox_after_cleaning[:5]:
        print(item)

Invalid bbox entries after cleaning: 0


## Class Distribution
Count how many bounding boxes belong to each class across the dataset.

In [8]:
from collections import Counter

class_counter = Counter()

for label_file in SORTED_DIR.glob("*/*/labels/*.txt"):
    lines = [line.strip() for line in label_file.read_text().splitlines() if line.strip()]

    for line in lines:
        parts = line.split()
        if len(parts) != 5:
            continue

        class_id = int(parts[0])
        class_counter[class_id] += 1

print("Bounding boxes per class:")
for class_id, count in sorted(class_counter.items()):
    print(f"  Class {class_id}: {count}")

Bounding boxes per class:
  Class 0: 26016
  Class 1: 11830
  Class 2: 10616
  Class 3: 10705
  Class 4: 6544


## Class Names
Human-readable class mapping used throughout the EDA notebook.

In [10]:
class_names = {
    0: "longitudinal crack",
    1: "transverse crack",
    2: "alligator crack",
    3: "other corruption",
    4: "Pothole"
}

for class_id, class_name in class_names.items():
    print(f"Class {class_id}: {class_name}")

Class 0: longitudinal crack
Class 1: transverse crack
Class 2: alligator crack
Class 3: other corruption
Class 4: Pothole


## Class Distribution by Split
Count bounding boxes for each damage class in train, val, and test.

In [11]:
from collections import Counter

split_class_counts = {}

for split in splits:
    counter = Counter()

    for label_file in (SORTED_DIR / split).glob("*/labels/*.txt"):
        lines = [line.strip() for line in label_file.read_text().splitlines() if line.strip()]

        for line in lines:
            parts = line.split()
            if len(parts) != 5:
                continue

            class_id = int(parts[0])
            counter[class_names[class_id]] += 1

    split_class_counts[split] = counter

for split, counter in split_class_counts.items():
    print(f"\n{split.upper()}:")
    for class_name, count in counter.items():
        print(f"  {class_name}: {count}")


TRAIN:
  longitudinal crack: 18201
  alligator crack: 7526
  Pothole: 4628
  transverse crack: 8386
  other corruption: 7554

VAL:
  longitudinal crack: 3890
  transverse crack: 1769
  alligator crack: 1553
  Pothole: 965
  other corruption: 1564

TEST:
  transverse crack: 1675
  longitudinal crack: 3925
  alligator crack: 1537
  Pothole: 951
  other corruption: 1587


## Empty vs Non-Empty Images by Country
Count how many images contain at least one annotation versus no annotation for each country.

In [12]:
for split in splits:
    print(f"\n{split.upper()}:")
    
    for country in countries:
        label_files = list((SORTED_DIR / split / country / "labels").glob("*.txt"))
        
        empty_count = 0
        non_empty_count = 0
        
        for label_file in label_files:
            lines = [line.strip() for line in label_file.read_text().splitlines() if line.strip()]
            if len(lines) == 0:
                empty_count += 1
            else:
                non_empty_count += 1
        
        total = empty_count + non_empty_count
        empty_ratio = empty_count / total if total else 0
        
        print(
            f"  {country}: "
            f"empty={empty_count}, "
            f"non_empty={non_empty_count}, "
            f"empty_ratio={empty_ratio:.2%}"
        )


TRAIN:
  China_Drone: empty=5, non_empty=1670, empty_ratio=0.30%
  China_MotorBike: empty=0, non_empty=1376, empty_ratio=0.00%
  Czech: empty=1216, non_empty=746, empty_ratio=61.98%
  India: empty=2691, non_empty=2677, empty_ratio=50.13%
  Japan: empty=535, non_empty=6897, empty_ratio=7.20%
  Norway: empty=3650, non_empty=2058, empty_ratio=63.95%
  United_States: empty=0, non_empty=3348, empty_ratio=0.00%

VAL:
  China_Drone: empty=0, non_empty=349, empty_ratio=0.00%
  China_MotorBike: empty=0, non_empty=296, empty_ratio=0.00%
  Czech: empty=271, non_empty=160, empty_ratio=62.88%
  India: empty=602, non_empty=570, empty_ratio=51.37%
  Japan: empty=136, non_empty=1414, empty_ratio=8.77%
  Norway: empty=828, non_empty=428, empty_ratio=65.92%
  United_States: empty=0, non_empty=704, empty_ratio=0.00%

TEST:
  China_Drone: empty=0, non_empty=377, empty_ratio=0.00%
  China_MotorBike: empty=0, non_empty=305, empty_ratio=0.00%
  Czech: empty=270, non_empty=166, empty_ratio=61.93%
  India: em

## Bounding Boxes by Country
Count the total number of annotated objects for each country in each split.

In [13]:
for split in splits:
    print(f"\n{split.upper()}:")
    
    for country in countries:
        total_boxes = 0
        label_files = list((SORTED_DIR / split / country / "labels").glob("*.txt"))
        
        for label_file in label_files:
            lines = [line.strip() for line in label_file.read_text().splitlines() if line.strip()]
            total_boxes += len(lines)
        
        print(f"  {country}: {total_boxes}")


TRAIN:
  China_Drone: 2656
  China_MotorBike: 3442
  Czech: 1216
  India: 5786
  Japan: 17529
  Norway: 7951
  United_States: 7715

VAL:
  China_Drone: 571
  China_MotorBike: 748
  Czech: 262
  India: 1249
  Japan: 3607
  Norway: 1683
  United_States: 1621

TEST:
  China_Drone: 613
  China_MotorBike: 737
  Czech: 267
  India: 1168
  Japan: 3617
  Norway: 1595
  United_States: 1678


## Class Distribution by Country
Count each damage class for every country across all splits.

In [14]:
from collections import Counter

for split in splits:
    print(f"\n{split.upper()}:")
    
    for country in countries:
        counter = Counter()
        label_files = list((SORTED_DIR / split / country / "labels").glob("*.txt"))
        
        for label_file in label_files:
            lines = [line.strip() for line in label_file.read_text().splitlines() if line.strip()]
            
            for line in lines:
                parts = line.split()
                if len(parts) != 5:
                    continue
                class_id = int(parts[0])
                counter[class_names[class_id]] += 1
        
        print(f"\n  {country}:")
        for class_name, count in counter.items():
            print(f"    {class_name}: {count}")


TRAIN:

  China_Drone:
    longitudinal crack: 982
    other corruption: 525
    alligator crack: 207
    transverse crack: 879
    Pothole: 63

  China_MotorBike:
    longitudinal crack: 1861
    transverse crack: 769
    other corruption: 189
    Pothole: 159
    alligator crack: 464

  Czech:
    longitudinal crack: 682
    Pothole: 133
    transverse crack: 275
    alligator crack: 126

  India:
    Pothole: 2236
    alligator crack: 1428
    longitudinal crack: 1103
    other corruption: 969
    transverse crack: 50

  Japan:
    alligator crack: 4394
    transverse crack: 2847
    other corruption: 5871
    longitudinal crack: 2781
    Pothole: 1636

  Norway:
    longitudinal crack: 6080
    alligator crack: 334
    Pothole: 311
    transverse crack: 1226

  United_States:
    longitudinal crack: 4712
    alligator crack: 573
    transverse crack: 2340
    Pothole: 90

VAL:

  China_Drone:
    longitudinal crack: 212
    other corruption: 119
    transverse crack: 184
    allig

## Objects Per Image
Measure how many bounding boxes appear in each labeled image.

In [ ]:
box_count_distribution = {}

for split in splits:
    counts = []
    label_files = list((SORTED_DIR / split).glob("*/labels/*.txt"))
    
    for label_file in label_files:
        lines = [line.strip() for line in label_file.read_text().splitlines() if line.strip()]
        counts.append(len(lines))
    
    box_count_distribution[split] = counts

for split, counts in box_count_distribution.items():
    total_images = len(counts)
    zero_boxes = sum(1 for c in counts if c == 0)
    one_box = sum(1 for c in counts if c == 1)
    two_or_more = sum(1 for c in counts if c >= 2)
    max_boxes = max(counts) if counts else 0
    avg_boxes = sum(counts) / total_images if total_images else 0

    print(f"\n{split.upper()}:")
    print(f"total images   : {total_images}")
    print(f"zero boxes     : {zero_boxes}")
    print(f"one box        : {one_box}")
    print(f"two or more    : {two_or_more}")
    print(f"avg boxes/img  : {avg_boxes:.3f}")
    print(f"max boxes/img  : {max_boxes}")


TRAIN:
  total images   : 26869
  zero boxes     : 8097
  one box        : 7165
  two or more    : 11607
  avg boxes/img  : 1.723
  max boxes/img  : 44

VAL:
  total images   : 5758
  zero boxes     : 1837
  one box        : 1517
  two or more    : 2404
  avg boxes/img  : 1.692
  max boxes/img  : 34

TEST:
  total images   : 5758
  zero boxes     : 1790
  one box        : 1536
  two or more    : 2432
  avg boxes/img  : 1.680
  max boxes/img  : 26


## Create Cleaned Dataset Directory
Create a final cleaned copy directory for YOLO-ready preprocessing outputs.

In [16]:
CLEANED_DIR = PROJECT_DIR / "Data" / "RDD_SPLIT_BY_COUNTRY_CLEANED"

for split in splits:
    for country in countries:
        (CLEANED_DIR / split / country / "images").mkdir(parents=True, exist_ok=True)
        (CLEANED_DIR / split / country / "labels").mkdir(parents=True, exist_ok=True)

print("CLEANED_DIR created:", CLEANED_DIR)

for split in splits:
    print(f"\n{split}:")
    print(sorted([p.name for p in (CLEANED_DIR / split).iterdir()]))

CLEANED_DIR created: /Users/ahadm/Desktop/computer-vision-project-sabiq/Data/RDD_SPLIT_BY_COUNTRY_CLEANED

train:
['China_Drone', 'China_MotorBike', 'Czech', 'India', 'Japan', 'Norway', 'United_States']

val:
['China_Drone', 'China_MotorBike', 'Czech', 'India', 'Japan', 'Norway', 'United_States']

test:
['China_Drone', 'China_MotorBike', 'Czech', 'India', 'Japan', 'Norway', 'United_States']


## Copy Cleaned Dataset
Copy the validated and cleaned dataset into the final YOLO-ready directory.

In [17]:
import shutil

copied_images = 0
copied_labels = 0

for split in splits:
    for country in countries:
        src_img_dir = SORTED_DIR / split / country / "images"
        src_lbl_dir = SORTED_DIR / split / country / "labels"

        dst_img_dir = CLEANED_DIR / split / country / "images"
        dst_lbl_dir = CLEANED_DIR / split / country / "labels"

        for image_path in src_img_dir.iterdir():
            if image_path.suffix in image_suffixes:
                shutil.copy2(image_path, dst_img_dir / image_path.name)
                copied_images += 1

        for label_path in src_lbl_dir.glob("*.txt"):
            shutil.copy2(label_path, dst_lbl_dir / label_path.name)
            copied_labels += 1

print("Copied images:", copied_images)
print("Copied labels:", copied_labels)

Copied images: 38385
Copied labels: 38385


## Final Verification of Cleaned Dataset
Check that the final cleaned dataset contains matching image and label counts for every split and country.

In [18]:
for split in splits:
    print(f"\n{split.upper()}:")
    
    for country in countries:
        img_dir = CLEANED_DIR / split / country / "images"
        lbl_dir = CLEANED_DIR / split / country / "labels"

        n_images = len([p for p in img_dir.iterdir() if p.suffix in image_suffixes])
        n_labels = len(list(lbl_dir.glob("*.txt")))

        print(f"  {country}: images={n_images}, labels={n_labels}")


TRAIN:
  China_Drone: images=1675, labels=1675
  China_MotorBike: images=1376, labels=1376
  Czech: images=1962, labels=1962
  India: images=5368, labels=5368
  Japan: images=7432, labels=7432
  Norway: images=5708, labels=5708
  United_States: images=3348, labels=3348

VAL:
  China_Drone: images=349, labels=349
  China_MotorBike: images=296, labels=296
  Czech: images=431, labels=431
  India: images=1172, labels=1172
  Japan: images=1550, labels=1550
  Norway: images=1256, labels=1256
  United_States: images=704, labels=704

TEST:
  China_Drone: images=377, labels=377
  China_MotorBike: images=305, labels=305
  Czech: images=436, labels=436
  India: images=1166, labels=1166
  Japan: images=1524, labels=1524
  Norway: images=1197, labels=1197
  United_States: images=753, labels=753


## Image Size Analysis
Inspect image width and height across the dataset.

In [ ]:
from PIL import Image

image_size_stats = {}

for split in splits:
    sizes = []

    for image_path in (CLEANED_DIR / split).glob("*/images/*"):
        if image_path.suffix not in image_suffixes:
            continue

        with Image.open(image_path) as img:
            width, height = img.size
            sizes.append((width, height))

    widths = [w for w, h in sizes]
    heights = [h for w, h in sizes]

    image_size_stats[split] = sizes

    print(f"\n{split.upper()}:")
    print(f"total images : {len(sizes)}")
    print(f"min width    : {min(widths)}")
    print(f"max width    : {max(widths)}")
    print(f"min height   : {min(heights)}")
    print(f"max height   : {max(heights)}")
    print(f"unique sizes : {len(set(sizes))}")
    print(f"sample sizes : {list(sorted(set(sizes)))[:10]}")


TRAIN:
  total images : 26869
  min width    : 512
  max width    : 4040
  min height   : 512
  max height   : 2044
  unique sizes : 10
  sample sizes : [(512, 512), (540, 540), (600, 600), (640, 640), (720, 720), (1024, 1024), (1080, 1080), (3643, 2041), (3650, 2044), (4040, 2035)]

VAL:
  total images : 5758
  min width    : 512
  max width    : 4040
  min height   : 512
  max height   : 2044
  unique sizes : 10
  sample sizes : [(512, 512), (540, 540), (600, 600), (640, 640), (720, 720), (1024, 1024), (1080, 1080), (3643, 2041), (3650, 2044), (4040, 2035)]

TEST:
  total images : 5758
  min width    : 512
  max width    : 4040
  min height   : 512
  max height   : 2044
  unique sizes : 10
  sample sizes : [(512, 512), (540, 540), (600, 600), (640, 640), (720, 720), (1024, 1024), (1080, 1080), (3643, 2041), (3650, 2044), (4040, 2035)]


## Image Size Distribution
Count how many images belong to each resolution in each split.

In [24]:
from collections import Counter
from PIL import Image

for split in splits:
    size_counter = Counter()

    for image_path in (CLEANED_DIR / split).glob("*/images/*"):
        if image_path.suffix not in image_suffixes:
            continue

        with Image.open(image_path) as img:
            size_counter[img.size] += 1

    print(f"\n{split.upper()}:")
    for size, count in sorted(size_counter.items()):
        print(f"  {size}: {count}")


TRAIN:
  (512, 512): 3051
  (540, 540): 86
  (600, 600): 9161
  (640, 640): 3348
  (720, 720): 5368
  (1024, 1024): 111
  (1080, 1080): 36
  (3643, 2041): 2032
  (3650, 2044): 656
  (4040, 2035): 3020

VAL:
  (512, 512): 645
  (540, 540): 23
  (600, 600): 1934
  (640, 640): 704
  (720, 720): 1172
  (1024, 1024): 18
  (1080, 1080): 6
  (3643, 2041): 429
  (3650, 2044): 136
  (4040, 2035): 691

TEST:
  (512, 512): 682
  (540, 540): 15
  (600, 600): 1921
  (640, 640): 753
  (720, 720): 1166
  (1024, 1024): 18
  (1080, 1080): 6
  (3643, 2041): 435
  (3650, 2044): 131
  (4040, 2035): 631


## Image Resolution Ratio
Show the percentage of images for each resolution in each split.

In [25]:
from collections import Counter
from PIL import Image

for split in splits:
    size_counter = Counter()

    for image_path in (CLEANED_DIR / split).glob("*/images/*"):
        if image_path.suffix not in image_suffixes:
            continue

        with Image.open(image_path) as img:
            size_counter[img.size] += 1

    total = sum(size_counter.values())

    print(f"\n{split.upper()}:")
    for size, count in sorted(size_counter.items()):
        ratio = count / total
        print(f"  {size}: {count} ({ratio:.2%})")


TRAIN:
  (512, 512): 3051 (11.36%)
  (540, 540): 86 (0.32%)
  (600, 600): 9161 (34.10%)
  (640, 640): 3348 (12.46%)
  (720, 720): 5368 (19.98%)
  (1024, 1024): 111 (0.41%)
  (1080, 1080): 36 (0.13%)
  (3643, 2041): 2032 (7.56%)
  (3650, 2044): 656 (2.44%)
  (4040, 2035): 3020 (11.24%)

VAL:
  (512, 512): 645 (11.20%)
  (540, 540): 23 (0.40%)
  (600, 600): 1934 (33.59%)
  (640, 640): 704 (12.23%)
  (720, 720): 1172 (20.35%)
  (1024, 1024): 18 (0.31%)
  (1080, 1080): 6 (0.10%)
  (3643, 2041): 429 (7.45%)
  (3650, 2044): 136 (2.36%)
  (4040, 2035): 691 (12.00%)

TEST:
  (512, 512): 682 (11.84%)
  (540, 540): 15 (0.26%)
  (600, 600): 1921 (33.36%)
  (640, 640): 753 (13.08%)
  (720, 720): 1166 (20.25%)
  (1024, 1024): 18 (0.31%)
  (1080, 1080): 6 (0.10%)
  (3643, 2041): 435 (7.55%)
  (3650, 2044): 131 (2.28%)
  (4040, 2035): 631 (10.96%)


## EDA Summary

The dataset was successfully validated, reorganized by country, and cleaned for YOLO-based road damage detection.

### Key findings
- The dataset contains **38,385 images** and **38,385 label files** across `train`, `val`, and `test`.
- The split structure is consistent, and image-label matching is correct across all subsets.
- About **30% of images are negative samples** (empty labels), and this ratio is consistent across `train`, `val`, and `test`.
- Annotation quality is very good overall:
  - no bad formatting issues
  - no invalid class IDs
  - only **one invalid bounding box** was found and removed successfully
- The dataset includes **5 classes**:
  - longitudinal crack
  - transverse crack
  - alligator crack
  - other corruption
  - pothole
- Class distribution is **imbalanced**, with `longitudinal crack` being the most frequent and `pothole` the least frequent.
- Data characteristics vary significantly by country:
  - `Japan` contains the highest number of annotations
  - `Czech`, `India`, and `Norway` contain many negative samples
  - some classes are clearly dominant in specific countries
- Image resolutions are mixed but consistent across splits:
  - most images are `600x600`, `720x720`, `640x640`, and `512x512`
  - a smaller portion of the dataset contains high-resolution wide images such as `4040x2035`

### Cleaning decisions
- Empty-label images were **kept**, since they are useful for training object detection models to distinguish damage from non-damage scenes.
- No manual image resizing was applied during preprocessing.
- A final cleaned dataset directory was created: `RDD_SPLIT_BY_COUNTRY_CLEANED`
